In [1]:
import pandas as pd
import numpy as np
import re

# Load
customer = pd.read_csv('KartZone_Customers.csv')

# Data Understanding
print("No of rows    :", customer.shape[0])
print("No of Columns :", customer.shape[1])
print()

print(customer.head())
print()

print(customer.tail())
print()

print(customer.info())
print()

print(customer.describe())
print()

print(customer.columns.tolist())

No of rows    : 1035
No of Columns : 17

  Customer_ID   Customer_Name   Age  Gender       City        State Region  \
0       C1001   Pallavi Gupta  36.0    Male   chennai    Tamil Nadu  South   
1       C1002    Shreya Reddy -10.0       F     MUMBAI  maharashtra   West   
2       C1003     PALLAVI DAS  32.0    Male        Hyd    Telangana  South   
3       C1004    Shreya Patel  40.0       f  Hyderabad          NaN  south   
4       C1005     Aarav Dubey  35.0  FEMALE     mumbai  Maharashtra   West   

  Customer_Segment Registration_Date Registration_Source Last_Login_Date  \
0          regular        14/05/2022                 app      07/03/2023   
1              New        09/25/2024                 NaN      05-06-2024   
2              New        05-08-2023        instagram ad             NaN   
3         Newcomer        28-04-2023                 APP      01/12/2024   
4          PREMIUM        21-03-2020             Website             NaN   

  Last_Order_Date                

In [2]:
# Remove completely blank rows first
print("Before removing blank rows:", len(customer))

customer = customer.dropna(how='all').copy()

print("After removing blank rows:", len(customer))
print("Completely blank rows remaining:", customer.isna().all(axis=1).sum())

# Remove exact duplicate rows
print("\nExact duplicates before removal:", customer.duplicated().sum())

customer = customer.drop_duplicates().copy()
customer = customer.reset_index(drop=True)

print("Exact duplicates after removal:", customer.duplicated().sum())
print("Final rows after structural cleaning:", len(customer))

Before removing blank rows: 1035
After removing blank rows: 1020
Completely blank rows remaining: 0

Exact duplicates before removal: 20
Exact duplicates after removal: 0
Final rows after structural cleaning: 1000


In [3]:
import re
# Clean Customer_Name

def clean_customer_name(val):
    if pd.isna(val):
        return np.nan

    name = str(val).strip()

    # Treat placeholder text as missing
    if name.lower() in ['nan', 'none', 'na', 'n/a', 'null', '-', '']:
        return np.nan

    # Remove title prefix added by generator
    name = re.sub(r'^(mr\.?\s+|mrs\.?\s+|ms\.?\s+)', '', name,
                  flags=re.IGNORECASE)

    # Normalize spacing and case
    name = re.sub(r'\s+', ' ', name).strip()
    return name.title()

customer['Customer_Name'] = customer['Customer_Name'].apply(
    clean_customer_name
)

print("Missing Customer_Name:", customer['Customer_Name'].isna().sum())
print("\nSample cleaned names:")
print(customer[['Customer_ID', 'Customer_Name']].head(10))

Missing Customer_Name: 0

Sample cleaned names:
  Customer_ID  Customer_Name
0       C1001  Pallavi Gupta
1       C1002   Shreya Reddy
2       C1003    Pallavi Das
3       C1004   Shreya Patel
4       C1005    Aarav Dubey
5       C1006     Vijay Nair
6       C1007    Manish Iyer
7       C1008     Nisha Iyer
8       C1009    Priya Naidu
9       C1010   Neeraj Reddy


In [4]:
# Clean and Validate Age

# Convert Age to numeric
customer['Age'] = pd.to_numeric(customer['Age'], errors='coerce')

# Invalid ages: outside realistic customer range
invalid_age = (
    (customer['Age'] < 18) |
    (customer['Age'] > 80)
)

print("Invalid ages:", invalid_age.sum())
print("Missing ages before treatment:", customer['Age'].isna().sum())

# Replace invalid ages with missing
customer.loc[invalid_age, 'Age'] = np.nan

# Fill missing ages using Customer_Segment median
customer['Age'] = customer.groupby('Customer_Segment')['Age'].transform(
    lambda x: x.fillna(x.median())
)

# Safety fallback: overall median
customer['Age'] = customer['Age'].fillna(customer['Age'].median()).round().astype(int)

print("\nAge summary after cleaning:")
print(customer['Age'].describe())

print("\nMissing ages after treatment:", customer['Age'].isna().sum())

Invalid ages: 154
Missing ages before treatment: 60

Age summary after cleaning:
count    1000.000000
mean       32.310000
std         8.651401
min        18.000000
25%        26.000000
50%        32.000000
75%        38.000000
max        65.000000
Name: Age, dtype: float64

Missing ages after treatment: 0


In [5]:
#Standardize Gender

In [6]:
# Standardize Gender

def clean_gender(val):
    if pd.isna(val):
        return 'Unknown'
    
    val = str(val).strip().lower()
    
    if val in ['male', 'm', '1']:
        return 'Male'
    elif val in ['female', 'f', '2']:
        return 'Female'
    else:
        return 'Unknown'

customer['Gender'] = customer['Gender'].apply(clean_gender)

print("Gender distribution:")
print(customer['Gender'].value_counts())

print("\nMissing Gender:", customer['Gender'].isna().sum())

Gender distribution:
Gender
Female    520
Male      480
Name: count, dtype: int64

Missing Gender: 0


In [7]:
# STEP 7 — Standardize City

def standardize_city(val):
    if pd.isna(val):
        return np.nan

    val = str(val).strip().lower()

    city_map = {
        'mumbai': 'Mumbai',
        'bombay': 'Mumbai',

        'delhi': 'Delhi',
        'new delhi': 'Delhi',

        'bangalore': 'Bangalore',
        'bengaluru': 'Bangalore',
        'banglore': 'Bangalore',

        'chennai': 'Chennai',
        'madras': 'Chennai',

        'hyderabad': 'Hyderabad',
        'hyd': 'Hyderabad',
        'hydrabad': 'Hyderabad',

        'pune': 'Pune',
        'poona': 'Pune'
    }

    return city_map.get(val, np.nan)


customer['City'] = customer['City'].apply(standardize_city)

print("City distribution after standardization:")
print(customer['City'].value_counts(dropna=False))

print("\nMissing City:", customer['City'].isna().sum())

City distribution after standardization:
City
Mumbai       226
Delhi        205
Bangalore    205
Chennai      126
Hyderabad    112
Pune          69
NaN           57
Name: count, dtype: int64

Missing City: 57


In [8]:
# STEP 8 — Standardize State and Region from City

city_info = {
    'Mumbai':    ('Maharashtra', 'West'),
    'Delhi':     ('Delhi', 'North'),
    'Bangalore': ('Karnataka', 'South'),
    'Chennai':   ('Tamil Nadu', 'South'),
    'Hyderabad': ('Telangana', 'South'),
    'Pune':      ('Maharashtra', 'West')
}

# Derive clean State from City
customer['State'] = customer['City'].map(
    lambda x: city_info.get(x, (np.nan, np.nan))[0]
)

# Derive clean Region from City
customer['Region'] = customer['City'].map(
    lambda x: city_info.get(x, (np.nan, np.nan))[1]
)

print("State distribution:")
print(customer['State'].value_counts(dropna=False))

print("\nRegion distribution:")
print(customer['Region'].value_counts(dropna=False))

print("\nMissing State:", customer['State'].isna().sum())
print("Missing Region:", customer['Region'].isna().sum())

State distribution:
State
Maharashtra    295
Delhi          205
Karnataka      205
Tamil Nadu     126
Telangana      112
NaN             57
Name: count, dtype: int64

Region distribution:
Region
South    443
West     295
North    205
NaN       57
Name: count, dtype: int64

Missing State: 57
Missing Region: 57


In [9]:
# STEP 9 — Standardize Customer Segment

def standardize_segment(val):
    if pd.isna(val):
        return np.nan

    val = str(val).strip().lower()

    segment_map = {
        'premium': 'Premium',
        'vip': 'Premium',

        'regular': 'Regular',
        'standard': 'Regular',

        'new': 'New',
        'newcomer': 'New',

        'at-risk': 'At-Risk',
        'atrisk': 'At-Risk',
        'at risk': 'At-Risk',

        'churned': 'Churned',
        'inactive': 'Churned'
    }

    return segment_map.get(val, np.nan)


customer['Customer_Segment'] = customer['Customer_Segment'].apply(
    standardize_segment
)

print("Customer Segment distribution before filling:")
print(customer['Customer_Segment'].value_counts(dropna=False))

print("\nMissing Customer_Segment:", customer['Customer_Segment'].isna().sum())

# Fill any missing values with the most common segment
customer['Customer_Segment'] = customer['Customer_Segment'].fillna(
    customer['Customer_Segment'].mode()[0]
)

print("\nFinal Customer Segment distribution:")
print(customer['Customer_Segment'].value_counts())

print("\nMissing Customer_Segment after cleaning:",
      customer['Customer_Segment'].isna().sum())

Customer Segment distribution before filling:
Customer_Segment
Regular    379
New        284
At-Risk    127
Premium    121
Churned     89
Name: count, dtype: int64

Missing Customer_Segment: 0

Final Customer Segment distribution:
Customer_Segment
Regular    379
New        284
At-Risk    127
Premium    121
Churned     89
Name: count, dtype: int64

Missing Customer_Segment after cleaning: 0


In [10]:
# STEP 10 — Standardize Registration Source

def standardize_source(val):
    if pd.isna(val):
        return 'Unknown'

    val = str(val).strip().lower()

    source_map = {
        'app': 'App',
        'mobile app': 'App',

        'website': 'Website',
        'web': 'Website',

        'referral': 'Referral',
        'refer': 'Referral',

        'instagram ad': 'Instagram Ad',
        'insta ad': 'Instagram Ad',

        'google ad': 'Google Ad',
        'google': 'Google Ad',

        'organic': 'Organic',
        'direct': 'Organic'
    }

    return source_map.get(val, 'Unknown')


customer['Registration_Source'] = customer['Registration_Source'].apply(
    standardize_source
)

print("Registration Source distribution:")
print(customer['Registration_Source'].value_counts())

print("\nUnknown sources:",
      (customer['Registration_Source'] == 'Unknown').sum())

Registration Source distribution:
Registration_Source
App             267
Website         184
Instagram Ad    161
Referral        131
Google Ad       110
Organic          82
Unknown          65
Name: count, dtype: int64

Unknown sources: 65


In [11]:
# STEP 11 — Parse and Validate Date Columns

from datetime import datetime

def parse_date(val):
    if pd.isna(val):
        return pd.NaT

    val = str(val).strip()

    if val.lower() in ['', 'na', 'n/a', 'nan', 'null', '-']:
        return pd.NaT

    formats = [
        '%Y-%m-%d',
        '%d/%m/%Y',
        '%d-%m-%Y',
        '%m/%d/%Y',
        '%d.%m.%Y',
        '%Y/%m/%d'
    ]

    for fmt in formats:
        try:
            return pd.to_datetime(
                datetime.strptime(val, fmt)
            )
        except ValueError:
            continue

    return pd.NaT


# Apply to all date columns
date_cols = [
    'Registration_Date',
    'Last_Login_Date',
    'Last_Order_Date'
]

for col in date_cols:
    customer[col] = customer[col].apply(parse_date)


print("Date column data types:")
print(customer[date_cols].dtypes)

print("\nMissing dates:")
print(customer[date_cols].isna().sum())

# Validation checks
invalid_login = (
    customer['Last_Login_Date'].notna() &
    (customer['Last_Login_Date'] < customer['Registration_Date'])
)

invalid_order = (
    customer['Last_Order_Date'].notna() &
    (customer['Last_Order_Date'] < customer['Registration_Date'])
)

print("\nLast login before registration:", invalid_login.sum())
print("Last order before registration:", invalid_order.sum())

Date column data types:
Registration_Date    datetime64[ns]
Last_Login_Date      datetime64[ns]
Last_Order_Date      datetime64[ns]
dtype: object

Missing dates:
Registration_Date      0
Last_Login_Date      194
Last_Order_Date      220
dtype: int64

Last login before registration: 137
Last order before registration: 147


In [12]:
# STEP 12 — Fix Invalid Date Relationships

# Last login cannot be before registration
invalid_login = (
    customer['Last_Login_Date'].notna() &
    (customer['Last_Login_Date'] < customer['Registration_Date'])
)

customer.loc[invalid_login, 'Last_Login_Date'] = pd.NaT


# Last order cannot be before registration
invalid_order = (
    customer['Last_Order_Date'].notna() &
    (customer['Last_Order_Date'] < customer['Registration_Date'])
)

customer.loc[invalid_order, 'Last_Order_Date'] = pd.NaT


# Final validation
print("Invalid Last_Login_Date values fixed:", invalid_login.sum())
print("Invalid Last_Order_Date values fixed:", invalid_order.sum())

print("\nMissing dates after treatment:")
print(customer[
    ['Registration_Date', 'Last_Login_Date', 'Last_Order_Date']
].isna().sum())

# Confirm no invalid relationships remain
print("\nRemaining invalid login dates:",
      (
          customer['Last_Login_Date'].notna() &
          (customer['Last_Login_Date'] < customer['Registration_Date'])
      ).sum()
)

print("Remaining invalid order dates:",
      (
          customer['Last_Order_Date'].notna() &
          (customer['Last_Order_Date'] < customer['Registration_Date'])
      ).sum())

Invalid Last_Login_Date values fixed: 137
Invalid Last_Order_Date values fixed: 147

Missing dates after treatment:
Registration_Date      0
Last_Login_Date      331
Last_Order_Date      367
dtype: int64

Remaining invalid login dates: 0
Remaining invalid order dates: 0


In [13]:
# STEP 13 — Validate and Clean Phone and Email


# STEP 13 — Validate and Clean Phone

import re

def clean_phone(val):
    if pd.isna(val):
        return np.nan

    val = str(val).strip()

    # Treat placeholders as missing
    if val.lower() in ['', 'na', 'n/a', 'nan', 'null', '-', 'xxxxxxxxxx',
                       '0000000000', '123']:
        return np.nan

    # Remove +, spaces, hyphens and brackets
    val = re.sub(r'[\+\-\s\(\)]', '', val)

    # Remove Indian country code
    if val.startswith('0091'):
        val = val[4:]
    elif val.startswith('91') and len(val) == 12:
        val = val[2:]

    # Valid Indian mobile: 10 digits starting from 6–9
    if re.fullmatch(r'[6-9]\d{9}', val):
        return val

    return np.nan


customer['Phone'] = customer['Phone'].apply(clean_phone)

print("Valid phones:", customer['Phone'].notna().sum())
print("Invalid/Missing phones:", customer['Phone'].isna().sum())
print(f"Valid phone percentage: "
      f"{customer['Phone'].notna().mean() * 100:.1f}%")


# STEP 14 — Validate and Clean Email

def clean_email(val):
    if pd.isna(val):
        return np.nan

    val = str(val).strip().lower()

    # Treat placeholders as missing
    if val in ['', 'na', 'n/a', 'nan', 'null', '-']:
        return np.nan

    # Basic email validation
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'

    if re.fullmatch(pattern, val):
        return val

    return np.nan


customer['Email'] = customer['Email'].apply(clean_email)

print("\nValid emails:", customer['Email'].notna().sum())
print("Invalid/Missing emails:", customer['Email'].isna().sum())
print(f"Valid email percentage: "
      f"{customer['Email'].notna().mean() * 100:.1f}%")


Valid phones: 674
Invalid/Missing phones: 326
Valid phone percentage: 67.4%

Valid emails: 585
Invalid/Missing emails: 415
Valid email percentage: 58.5%


In [14]:
# STEP 15 — Clean Loyalty Score

# Convert to numeric
customer['Loyalty_Score'] = pd.to_numeric(
    customer['Loyalty_Score'],
    errors='coerce'
)

# Invalid scores: outside valid range 1–100
invalid_loyalty = (
    (customer['Loyalty_Score'] < 1) |
    (customer['Loyalty_Score'] > 100)
)

print("Invalid Loyalty Scores:", invalid_loyalty.sum())
print("Missing Loyalty Scores before treatment:",
      customer['Loyalty_Score'].isna().sum())

# Replace invalid values with missing
customer.loc[invalid_loyalty, 'Loyalty_Score'] = np.nan

# Fill missing values using Customer Segment median
customer['Loyalty_Score'] = customer.groupby(
    'Customer_Segment'
)['Loyalty_Score'].transform(
    lambda x: x.fillna(x.median())
)

# Safety fallback: overall median
customer['Loyalty_Score'] = customer['Loyalty_Score'].fillna(
    customer['Loyalty_Score'].median()
)

# Round to whole numbers
customer['Loyalty_Score'] = customer['Loyalty_Score'].round().astype(int)

print("\nLoyalty Score by Customer Segment:")
print(
    customer.groupby('Customer_Segment')['Loyalty_Score']
    .describe()
    .round(2)
)

print("\nMissing Loyalty Scores after treatment:",
      customer['Loyalty_Score'].isna().sum())

print("Final Loyalty Score range:",
      customer['Loyalty_Score'].min(),
      "to",
      customer['Loyalty_Score'].max())

Invalid Loyalty Scores: 0
Missing Loyalty Scores before treatment: 119

Loyalty Score by Customer Segment:
                  count   mean    std   min   25%   50%   75%    max
Customer_Segment                                                    
At-Risk           127.0  27.38  12.53   6.0  17.0  28.0  37.0   50.0
Churned            89.0  15.81   9.22   1.0   8.0  16.0  22.0   35.0
New               284.0  33.42  12.10  10.0  23.0  34.0  42.0   55.0
Premium           121.0  75.84  10.66  57.0  67.0  75.0  82.0  100.0
Regular           379.0  56.13  15.55  30.0  42.0  56.0  68.5   85.0

Missing Loyalty Scores after treatment: 0
Final Loyalty Score range: 1 to 100


In [15]:
# STEP 16 — Standardize Newsletter Subscribed

def standardize_bool(val):
    if pd.isna(val):
        return 'Unknown'

    val = str(val).strip().lower()

    if val in ['yes', '1', 'true', 'y']:
        return 'Yes'
    elif val in ['no', '0', 'false', 'n']:
        return 'No'
    else:
        return 'Unknown'


customer['Newsletter_Subscribed'] = customer[
    'Newsletter_Subscribed'
].apply(standardize_bool)

print("Newsletter Subscription distribution:")
print(customer['Newsletter_Subscribed'].value_counts())

print("\nMissing Newsletter values:",
      customer['Newsletter_Subscribed'].isna().sum())


# STEP 17 — Standardize Preferred Category

def standardize_category(val):
    if pd.isna(val):
        return 'Unknown'

    val = str(val).strip().lower()

    if val in ['electronics', 'electronic']:
        return 'Electronics'
    elif val == 'fashion':
        return 'Fashion'
    elif val in ['home & kitchen', 'home and kitchen', 'home', 'kitchen']:
        return 'Home & Kitchen'
    elif val in ['beauty', 'personal care']:
        return 'Beauty'
    else:
        return 'Unknown'


customer['Preferred_Category'] = customer[
    'Preferred_Category'
].apply(standardize_category)

print("\nPreferred Category distribution:")
print(customer['Preferred_Category'].value_counts())

print("\nMissing Preferred Category values:",
      customer['Preferred_Category'].isna().sum())

Newsletter Subscription distribution:
Newsletter_Subscribed
Yes        454
No         444
Unknown    102
Name: count, dtype: int64

Missing Newsletter values: 0

Preferred Category distribution:
Preferred_Category
Electronics       267
Fashion           223
Home & Kitchen    190
Unknown           177
Beauty            143
Name: count, dtype: int64

Missing Preferred Category values: 0


In [16]:
# STEP 18 — Outlier Check and Treatment for Loyalty_Score

# IQR method
Q1 = customer['Loyalty_Score'].quantile(0.25)
Q3 = customer['Loyalty_Score'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Loyalty Score IQR bounds: {lower:.1f} to {upper:.1f}")

outlier_mask = (
    (customer['Loyalty_Score'] < lower) |
    (customer['Loyalty_Score'] > upper)
)

print(f"Outliers found: {outlier_mask.sum()}")

# Loyalty Score has a defined valid business range: 1–100
# Therefore, only enforce the valid range rather than
# unnecessarily capping legitimate high/low customer scores.
customer['Loyalty_Score'] = customer['Loyalty_Score'].clip(
    lower=1,
    upper=100
)

print("\nFinal Loyalty Score summary:")
print(customer['Loyalty_Score'].describe())

print("\nValues outside valid range:",
      ((customer['Loyalty_Score'] < 1) |
       (customer['Loyalty_Score'] > 100)).sum())

Loyalty Score IQR bounds: -19.0 to 109.0
Outliers found: 0

Final Loyalty Score summary:
count    1000.000000
mean       44.828000
std        22.127993
min         1.000000
25%        29.000000
50%        42.000000
75%        61.000000
max       100.000000
Name: Loyalty_Score, dtype: float64

Values outside valid range: 0


In [17]:
# STEP 19 — Feature Engineering

# Fixed analysis date for reproducibility
today = pd.Timestamp('2026-08-26')

# 1. Customer Tenure
customer['Customer_Tenure_Days'] = (
    today - customer['Registration_Date']
).dt.days

# 2. Days Since Last Login
customer['Days_Since_Login'] = (
    today - customer['Last_Login_Date']
).dt.days

# 3. Days Since Last Order
customer['Days_Since_Order'] = (
    today - customer['Last_Order_Date']
).dt.days


# 4. Age Group
def age_group(age):
    if pd.isna(age):
        return 'Unknown'
    elif age <= 25:
        return '18-25'
    elif age <= 35:
        return '26-35'
    elif age <= 45:
        return '36-45'
    else:
        return '46+'


customer['Age_Group'] = customer['Age'].apply(age_group)


# 5. Churn Flag
customer['Churn_Flag'] = customer['Customer_Segment'].isin(
    ['Churned', 'At-Risk']
).astype(int)


# 6. Is Dormant — no login for more than 90 days
customer['Is_Dormant'] = (
    customer['Days_Since_Login'] > 90
).fillna(False).astype(int)


# 7. Is Valid Contact
customer['Is_Valid_Contact'] = (
    customer['Phone'].notna() &
    customer['Email'].notna()
).astype(int)


# 8. Tenure Band
def tenure_band(days):
    if pd.isna(days):
        return 'Unknown'
    elif days < 90:
        return 'New (< 3 months)'
    elif days < 365:
        return 'Growing (3-12 months)'
    elif days < 730:
        return 'Established (1-2 years)'
    else:
        return 'Loyal (2+ years)'


customer['Tenure_Band'] = customer[
    'Customer_Tenure_Days'
].apply(tenure_band)


print(customer[
    ['Customer_Tenure_Days',
     'Days_Since_Login',
     'Days_Since_Order',
     'Age_Group',
     'Churn_Flag',
     'Is_Dormant',
     'Is_Valid_Contact',
     'Tenure_Band']
].head(10))

   Customer_Tenure_Days  Days_Since_Login  Days_Since_Order Age_Group  \
0                  1565            1268.0            1150.0     36-45   
1                   700               NaN               NaN     26-35   
2                  1117               NaN             630.0     26-35   
3                  1216             633.0               NaN     36-45   
4                  2349               NaN             818.0     26-35   
5                  2144             645.0             867.0     26-35   
6                   665             619.0             622.0       46+   
7                   790               NaN               NaN     26-35   
8                  1911               NaN               NaN     26-35   
9                  1125               NaN             919.0     36-45   

   Churn_Flag  Is_Dormant  Is_Valid_Contact              Tenure_Band  
0           0           1                 1         Loyal (2+ years)  
1           0           0                 1  Establish

In [18]:
# STEP 20 — Normalization and Transformation

from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Min-Max Normalization
scaler_mm_age = MinMaxScaler()
customer['Age_Normalized'] = scaler_mm_age.fit_transform(
    customer[['Age']]
)

scaler_mm_loyalty = MinMaxScaler()
customer['Loyalty_Normalized'] = scaler_mm_loyalty.fit_transform(
    customer[['Loyalty_Score']]
)


# Z-Score Standardization — Customer Tenure
scaler_z = StandardScaler()

customer['Tenure_Zscore'] = np.nan

valid_tenure = customer['Customer_Tenure_Days'].notna()

customer.loc[valid_tenure, 'Tenure_Zscore'] = (
    scaler_z.fit_transform(
        customer.loc[valid_tenure, ['Customer_Tenure_Days']]
    ).ravel()
)


# Log Transformation — Days Since Login
# Keep missing login information as missing
customer['Days_Since_Login_Log'] = np.where(
    customer['Days_Since_Login'].notna(),
    np.log1p(customer['Days_Since_Login']),
    np.nan
)


print(customer[
    ['Age_Normalized',
     'Loyalty_Normalized',
     'Tenure_Zscore',
     'Days_Since_Login_Log']
].describe())

       Age_Normalized  Loyalty_Normalized  Tenure_Zscore  Days_Since_Login_Log
count     1000.000000         1000.000000   1.000000e+03            669.000000
mean         0.304468            0.442707  -3.730349e-17              6.814466
std          0.184072            0.223515   1.000500e+00              0.253392
min          0.000000            0.000000  -1.678129e+00              6.403574
25%          0.170213            0.282828  -8.851450e-01              6.602588
50%          0.297872            0.414141   4.606584e-02              6.803505
75%          0.425532            0.606061   9.036165e-01              7.000334
max          1.000000            1.000000   1.633853e+00              7.427739


In [19]:
# STEP 21 — Encoding Categorical Variables

# 1. Ordinal Encoding — Customer Segment
segment_order = {
    'Churned': 0,
    'At-Risk': 1,
    'New': 2,
    'Regular': 3,
    'Premium': 4
}

customer['Segment_Encoded'] = customer[
    'Customer_Segment'
].map(segment_order)


# 2. Binary Encoding — Gender
customer['Gender_Encoded'] = customer[
    'Gender'
].map({
    'Male': 1,
    'Female': 0,
    'Unknown': -1
})


# 3. One-Hot Encoding — Region
region_dummies = pd.get_dummies(
    customer['Region'],
    prefix='Region',
    drop_first=True,
    dtype=int
)

customer = pd.concat(
    [customer, region_dummies],
    axis=1
)


# 4. One-Hot Encoding — Preferred Category
cat_dummies = pd.get_dummies(
    customer['Preferred_Category'],
    prefix='PrefCat',
    drop_first=True,
    dtype=int
)

customer = pd.concat(
    [customer, cat_dummies],
    axis=1
)


print(f"Shape after encoding: {customer.shape}")

print("\nEncoded columns:")
print([
    col for col in customer.columns
    if col.startswith('Region_') or
       col.startswith('PrefCat_') or
       col in ['Segment_Encoded', 'Gender_Encoded']
])

Shape after encoding: (1000, 37)

Encoded columns:
['Segment_Encoded', 'Gender_Encoded', 'Region_South', 'Region_West', 'PrefCat_Electronics', 'PrefCat_Fashion', 'PrefCat_Home & Kitchen', 'PrefCat_Unknown']



  KARTZONE CUSTOMERS — FINAL DATA QUALITY REPORT
  Total rows              : 1,000
  Total columns           : 37
  Remaining nulls         : 2,639
  Duplicate rows          : 0
  Duplicate Customer_IDs  : 0

  NULLS PER COLUMN:
    City                                  57 (5.7%)
    State                                 57 (5.7%)
    Region                                57 (5.7%)
    Last_Login_Date                      331 (33.1%)
    Last_Order_Date                      367 (36.7%)
    Email                                415 (41.5%)
    Phone                                326 (32.6%)
    Days_Since_Login                     331 (33.1%)
    Days_Since_Order                     367 (36.7%)
    Days_Since_Login_Log                 331 (33.1%)

  CUSTOMER SEGMENT DISTRIBUTION:
Customer_Segment
Regular    379
New        284
At-Risk    127
Premium    121
Churned     89
Name: count, dtype: int64

  CITY DISTRIBUTION:
City
Mumbai       226
Delhi        205
Bangalore    205
Chennai      

In [21]:
# STEP 23 — Location Consistency Check

print("Missing location values:")
print(customer[['City', 'State', 'Region']].isna().sum())

print("\nRows with missing location:")
print(
    customer[
        customer[['City', 'State', 'Region']].isna().any(axis=1)
    ][['Customer_ID', 'City', 'State', 'Region']]
)

Missing location values:
City      57
State     57
Region    57
dtype: int64

Rows with missing location:
    Customer_ID City State Region
9         C1010  NaN   NaN    NaN
23        C1024  NaN   NaN    NaN
38        C1039  NaN   NaN    NaN
46        C1047  NaN   NaN    NaN
61        C1062  NaN   NaN    NaN
84        C1085  NaN   NaN    NaN
88        C1089  NaN   NaN    NaN
123       C1124  NaN   NaN    NaN
143       C1144  NaN   NaN    NaN
144       C1145  NaN   NaN    NaN
146       C1147  NaN   NaN    NaN
157       C1158  NaN   NaN    NaN
159       C1160  NaN   NaN    NaN
168       C1169  NaN   NaN    NaN
179       C1180  NaN   NaN    NaN
189       C1190  NaN   NaN    NaN
204       C1205  NaN   NaN    NaN
224       C1225  NaN   NaN    NaN
227       C1228  NaN   NaN    NaN
236       C1237  NaN   NaN    NaN
246       C1247  NaN   NaN    NaN
247       C1248  NaN   NaN    NaN
248       C1249  NaN   NaN    NaN
251       C1252  NaN   NaN    NaN
272       C1273  NaN   NaN    NaN
279       

In [22]:
# STEP — Finalize Missing Location Values

customer['City'] = customer['City'].fillna('Unknown')
customer['State'] = customer['State'].fillna('Unknown')
customer['Region'] = customer['Region'].fillna('Unknown')

print("Missing location values after treatment:")
print(customer[['City', 'State', 'Region']].isna().sum())

print("\nLocation distribution:")
print(customer['City'].value_counts())

Missing location values after treatment:
City      0
State     0
Region    0
dtype: int64

Location distribution:
City
Mumbai       226
Delhi        205
Bangalore    205
Chennai      126
Hyderabad    112
Pune          69
Unknown       57
Name: count, dtype: int64


In [23]:
# STEP 22 — Final Customer Data Quality Validation

print("\n" + "="*60)
print("  KARTZONE CUSTOMERS — FINAL DATA QUALITY REPORT")
print("="*60)

print(f"  Total rows              : {len(customer):,}")
print(f"  Total columns           : {len(customer.columns)}")
print(f"  Remaining nulls         : {customer.isnull().sum().sum():,}")
print(f"  Duplicate rows          : {customer.duplicated().sum()}")
print(f"  Duplicate Customer_IDs  : {customer['Customer_ID'].duplicated().sum()}")

print("\n  NULLS PER COLUMN:")
null_counts = customer.isnull().sum()
for col, n in null_counts[null_counts > 0].items():
    print(f"    {col:<35} {n:>4} "
          f"({n/len(customer)*100:.1f}%)")

print("\n  CUSTOMER SEGMENT DISTRIBUTION:")
print(customer['Customer_Segment'].value_counts())

print("\n  CITY DISTRIBUTION:")
print(customer['City'].value_counts())

print("\n  AGE GROUP DISTRIBUTION:")
print(customer['Age_Group'].value_counts())

print("\n  TENURE BAND DISTRIBUTION:")
print(customer['Tenure_Band'].value_counts())

print("\n  NEWSLETTER DISTRIBUTION:")
print(customer['Newsletter_Subscribed'].value_counts())

print("\n  PREFERRED CATEGORY DISTRIBUTION:")
print(customer['Preferred_Category'].value_counts())

print("\n  CHURN FLAG:")
print(customer['Churn_Flag'].value_counts())

print("\n  DORMANCY FLAG:")
print(customer['Is_Dormant'].value_counts())

print("\n  VALID CONTACT FLAG:")
print(customer['Is_Valid_Contact'].value_counts())

print("\n  AGE SUMMARY:")
print(customer['Age'].describe().round(2))

print("\n  LOYALTY SCORE SUMMARY:")
print(customer['Loyalty_Score'].describe().round(2))

print("\n  TENURE SUMMARY:")
print(customer['Customer_Tenure_Days'].describe().round(2))

print("\n  DATE VALIDATION:")

print(
    "  Login before registration:",
    (
        customer['Last_Login_Date'].notna() &
        (customer['Last_Login_Date'] <
         customer['Registration_Date'])
    ).sum()
)

print(
    "  Order before registration:",
    (
        customer['Last_Order_Date'].notna() &
        (customer['Last_Order_Date'] <
         customer['Registration_Date'])
    ).sum()
)

print("\n" + "="*60)
print("  CUSTOMER VALIDATION COMPLETE")
print("="*60)


  KARTZONE CUSTOMERS — FINAL DATA QUALITY REPORT
  Total rows              : 1,000
  Total columns           : 37
  Remaining nulls         : 2,468
  Duplicate rows          : 0
  Duplicate Customer_IDs  : 0

  NULLS PER COLUMN:
    Last_Login_Date                      331 (33.1%)
    Last_Order_Date                      367 (36.7%)
    Email                                415 (41.5%)
    Phone                                326 (32.6%)
    Days_Since_Login                     331 (33.1%)
    Days_Since_Order                     367 (36.7%)
    Days_Since_Login_Log                 331 (33.1%)

  CUSTOMER SEGMENT DISTRIBUTION:
Customer_Segment
Regular    379
New        284
At-Risk    127
Premium    121
Churned     89
Name: count, dtype: int64

  CITY DISTRIBUTION:
City
Mumbai       226
Delhi        205
Bangalore    205
Chennai      126
Hyderabad    112
Pune          69
Unknown       57
Name: count, dtype: int64

  AGE GROUP DISTRIBUTION:
Age_Group
26-35    453
36-45    272
18-25    207

In [24]:
# Save cleaned version
customer.to_csv('KartZone_Customers_Clean.csv', index=False)
print("Saved → KartZone_Customers_Clean.csv ✓")

# Save only essential columns for SQL load
essential_cols = [
    'Customer_ID','Customer_Name','Age','Gender','City','State','Region',
    'Customer_Segment','Registration_Date','Registration_Source',
    'Last_Login_Date','Last_Order_Date','Email','Phone','Loyalty_Score',
    'Preferred_Category','Newsletter_Subscribed',
    'Customer_Tenure_Days','Days_Since_Login','Days_Since_Order',
    'Age_Group','Churn_Flag','Is_Dormant','Is_Valid_Contact','Tenure_Band'
]

customer[essential_cols].to_csv(
    'KartZone_Customers_Final.csv',
    index=False
)

print("Saved → KartZone_Customers_Final.csv ✓")

Saved → KartZone_Customers_Clean.csv ✓
Saved → KartZone_Customers_Final.csv ✓


In [ ]:
#Feature Engineering

today = pd.Timestamp.now()

# 1. Customer Tenure
customer['Customer_Tenure_Days'] = (
    today - customer['Registration_Date']
).dt.days

# 2. Days Since Last Login
customer['Days_Since_Login'] = (
    today - customer['Last_Login_Date']
).dt.days

# 3. Days Since Last Order
customer['Days_Since_Order'] = (
    today - customer['Last_Order_Date']
).dt.days

# 4. Age Group
def age_group(age):
    if pd.isnull(age): return 'Unknown'
    if age <= 25:   return '18-25'
    elif age <= 35: return '26-35'
    elif age <= 45: return '36-45'
    else:           return '46+'

customer['Age_Group'] = customer['Age'].apply(age_group)

# 5. Churn Flag
customer['Churn_Flag'] = customer['Customer_Segment'].isin(
    ['Churned','At-Risk']
).astype(int)

# 6. Is Dormant — no login in 90 days
customer['Is_Dormant'] = (
    customer['Days_Since_Login'] > 90
).astype(int)

# 7. Is Valid Contact
customer['Is_Valid_Contact'] = (
    customer['Phone'].notna() & customer['Email'].notna()
).astype(int)

# 8. Tenure Band
def tenure_band(days):
    if pd.isnull(days): return 'Unknown'
    if days < 90:    return 'New (< 3 months)'
    elif days < 365: return 'Growing (3-12 months)'
    elif days < 730: return 'Established (1-2 years)'
    else:            return 'Loyal (2+ years)'

customer['Tenure_Band'] = customer['Customer_Tenure_Days'].apply(tenure_band)

print(customer[['Age_Group','Churn_Flag','Is_Dormant',
                'Is_Valid_Contact','Tenure_Band']].head(10))

In [ ]:
# Normalization
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Min-Max Normalization — Age and Loyalty_Score
scaler_mm = MinMaxScaler()
customer['Age_Normalized'] = scaler_mm.fit_transform(
    customer[['Age']]
)
customer['Loyalty_Normalized'] = scaler_mm.fit_transform(
    customer[['Loyalty_Score']]
)

# Z-Score Standardization — Tenure
scaler_z = StandardScaler()
customer['Tenure_Zscore'] = scaler_z.fit_transform(
    customer[['Customer_Tenure_Days']].fillna(0)
)

# Log Transformation — Days Since Login (right skewed)
customer['Days_Since_Login_Log'] = np.log1p(
    customer['Days_Since_Login'].fillna(0)
)

print(customer[['Age_Normalized','Loyalty_Normalized',
                'Tenure_Zscore','Days_Since_Login_Log']].describe())

In [ ]:
# Encoding Categorical Variables

# 1. Label Encoding — Segment (ordinal)
segment_order = {'Churned':0,'At-Risk':1,'New':2,'Regular':3,'Premium':4}
customer['Segment_Encoded'] = customer['Customer_Segment'].map(segment_order)

# 2. Binary Encoding — Gender
customer['Gender_Encoded'] = customer['Gender'].map({'Male':1,'Female':0})

# 3. One Hot Encoding — Region
region_dummies = pd.get_dummies(
    customer['Region'],
    prefix='Region',
    drop_first=True
)
customer = pd.concat([customer, region_dummies], axis=1)

# 4. One Hot Encoding — Preferred Category
cat_dummies = pd.get_dummies(
    customer['Preferred_Category'],
    prefix='PrefCat',
    drop_first=True
)
customer = pd.concat([customer, cat_dummies], axis=1)

print(f"Shape after encoding: {customer.shape}")
print(customer[['Segment_Encoded','Gender_Encoded']].head())

In [ ]:
#Save Clean Data

# Save cleaned version
customer.to_csv('KartZone_Customers_Clean.csv', index=False)
print("Saved → KartZone_Customers_Clean.csv ✓")

# Save only essential columns for SQL load
essential_cols = [
    'Customer_ID','Customer_Name','Age','Gender','City','State','Region',
    'Customer_Segment','Registration_Date','Registration_Source',
    'Last_Login_Date','Last_Order_Date','Email','Phone','Loyalty_Score',
    'Preferred_Category','Newsletter_Subscribed',
    'Customer_Tenure_Days','Days_Since_Login','Days_Since_Order',
    'Age_Group','Churn_Flag','Is_Dormant','Is_Valid_Contact','Tenure_Band'
]
customer[essential_cols].to_csv('KartZone_Customers_Final.csv', index=False)
print("Saved → KartZone_Customers_Final.csv ✓")


In [ ]:
#Final Validation

print("\n" + "="*55)
print("  FINAL DATA QUALITY REPORT")
print("="*55)
print(f"  Total rows        : {len(customer):,}")
print(f"  Total columns     : {len(customer.columns)}")
print(f"  Remaining nulls   : {customer.isnull().sum().sum():,}")
print(f"  Duplicates        : {customer.duplicated().sum()}")
print(f"\n  NULLS PER COLUMN:")
for col in customer.columns:
    n = customer[col].isnull().sum()
    if n > 0:
        print(f"    {col:<35} {n:>4} ({n/len(customer)*100:.1f}%)")

print(f"\n  SEGMENT DISTRIBUTION:")
print(customer['Customer_Segment'].value_counts())
print(f"\n  CITY DISTRIBUTION:")
print(customer['City'].value_counts())
print(f"\n  AGE STATS:")
print(customer['Age'].describe())